# Optimización de Antenas 5G mediante Algoritmo Genético

La implementación del **Algoritmo Genético (GA)** utilizado para optimizar las potencias de las 100 antenas de una red 5G rural.

El objetivo es maximizar la función:

$$
fitness(P)=cobertura_{norm}(P)-\lambda\,interferencia_{norm}(P)
$$

donde cada solución representa las potencias de transmisión de las 100 antenas y cada potencia se encuentra en el intervalo $[0,1]$ W.


### Objetivo

El Algoritmo Genético busca encontrar una configuración de potencias que proporcione una buena cobertura de la zona rural y, al mismo tiempo, controle la interferencia.

Cada individuo de la población representa una configuración completa de las antenas:

$$
P=[P_1,P_2,\ldots,P_{100}]
$$

donde:

- $P_i$ es la potencia de la antena $i$.
- $0 \leq P_i \leq 1$ W.
- Existen 100 variables de decisión.
- La función `fitness` del proyecto determina la calidad de cada configuración.

El GA utiliza selección por torneo, cruce BLX-α, mutación gaussiana y elitismo.


## 1. Definición del modelo


In [2]:
using Random, Statistics, Printf, DelimitedFiles

RUTA_FITNESS = normpath(joinpath(@__DIR__, "..", "core", "fitness.jl"))

if !isfile(RUTA_FITNESS)
    error("No se encontró core/fitness.jl en: $RUTA_FITNESS")
end

include(RUTA_FITNESS)

const Fit = Main.Rural5GFitness
const Modelo = Fit.Rural5GModel

const N_VARS = Modelo.N_ANTENAS
const P_MAX = Modelo.P_MAX
const ANTENAS = Modelo.antenas
const LAMBDA_DEFAULT = Fit.LAMBDA_DEFAULT

println("Número de antenas: ", N_VARS)
println("Potencia máxima por antena: ", P_MAX, " W")
println("Lambda por defecto: ", LAMBDA_DEFAULT)
println("Número de posiciones: ", length(ANTENAS))


Número de antenas: 100
Potencia máxima por antena: 1.0 W
Lambda por defecto: 0.5
Número de posiciones: 100


### Representación de una solución

Una solución del problema es un vector de `Float64` de longitud 100. Por ejemplo:

```text
[P₁, P₂, P₃, ..., P₁₀₀]
```

La función objetivo recibe este vector y devuelve un valor de fitness. Cuanto mayor sea el fitness, mejor es la solución.


In [3]:
# Ejemplo de solución aleatoria válida
solucion_ejemplo = P_MAX .* rand(N_VARS)

println("Longitud de la solución: ", length(solucion_ejemplo))
println("Potencia mínima: ", minimum(solucion_ejemplo), " W")
println("Potencia máxima: ", maximum(solucion_ejemplo), " W")
println("Fitness de ejemplo: ", Fit.fitness(solucion_ejemplo; lambda=LAMBDA_DEFAULT))


Longitud de la solución: 100
Potencia mínima: 0.01823310130030198 W
Potencia máxima: 0.9951229665077437 W
Fitness de ejemplo: 0.8123360744610251


## 2. Algoritmo genético

### 2.1. Parámetros

In [23]:
Base.@kwdef struct ParamsGA
    tam_pob::Int        = 50
    k_torneo::Int       = 3
    prob_cruce::Float64 = 0.9
    alpha::Float64      = 0.3     # BLX-α
    prob_mut::Float64   = 0.02    # probabilidad de mutar CADA gen (≈ 2 de 100)
    sigma_rel::Float64  = 0.1     # σ de la mutación como fracción de P_MAX
    n_elite::Int        = 2
    max_evals::Int      = 50_000
end

ParamsGA

### 2.2. Operadores genéticos

#### Reparación

In [24]:
"Reparación: lleva cada potencia a [0, p_max]."
reparar!(x::Vector{Float64}, p_max::Float64) = clamp!(x, 0.0, p_max)

reparar! (generic function with 1 method)

#### Selección por torneo

In [25]:
"Selección por torneo: devuelve el índice del ganador (maximiza fitness)."
function torneo(rng::AbstractRNG, fit::Vector{Float64}, k::Int)
    mejor = rand(rng, 1:length(fit))
    for _ in 2:k
        c = rand(rng, 1:length(fit))
        if fit[c] > fit[mejor]
            mejor = c
        end
    end
    return mejor
end

torneo (generic function with 1 method)

#### Cruce BLX-α

In [26]:
"Cruce BLX-α: cada gen del hijo se muestrea en [min-α·d, max+α·d]."
function cruce_blx(rng::AbstractRNG, a::Vector{Float64}, b::Vector{Float64},
                   alpha::Float64, p_max::Float64)
    n = length(a)
    h1 = similar(a); h2 = similar(a)
    for i in 1:n
        lo, hi = minmax(a[i], b[i])
        d = hi - lo
        l = lo - alpha * d
        u = hi + alpha * d
        h1[i] = l + (u - l) * rand(rng)
        h2[i] = l + (u - l) * rand(rng)
    end
    reparar!(h1, p_max); reparar!(h2, p_max)
    return h1, h2
end

cruce_blx (generic function with 1 method)

#### Mutación gaussiana

In [27]:
"Mutación gaussiana por gen, seguida de reparación (los negativos quedan en 0 = antena apagada)."
function mutar!(rng::AbstractRNG, x::Vector{Float64}, prob::Float64,
                sigma_abs::Float64, p_max::Float64)
    for i in eachindex(x)
        if rand(rng) < prob
            x[i] += sigma_abs * randn(rng)
        end
    end
    reparar!(x, p_max)
    return x
end

mutar! (generic function with 1 method)

### 2.3. Función de prueba

Se incluye una función sustituta cuyo óptimo teórico se encuentra cuando todas las potencias son `P_MAX/2`. Esto permite comprobar que la lógica del GA funciona independientemente del modelo de cobertura.


In [28]:
"Función sustituta para probar la lógica del GA sin el modelo. Óptimo = 0 en P_MAX/2."
fitness_dummy(x::Vector{Float64}) = -sum((x .- 0.5 * P_MAX) .^ 2)

fitness_dummy (generic function with 1 method)

### 2.4. Ejecución del Algoritmo Genético

El procedimiento principal:

1. Genera la población inicial.
2. Evalúa el fitness de cada individuo.
3. Conserva los mejores individuos mediante elitismo.
4. Selecciona padres mediante torneo.
5. Aplica cruce BLX-α.
6. Aplica mutación gaussiana.
7. Evalúa los nuevos individuos.
8. Repite hasta alcanzar el presupuesto de evaluaciones.
9. Devuelve la mejor solución encontrada y el historial de convergencia.


In [29]:
struct ResultadoGA
    semilla::Int
    mejor_x::Vector{Float64}
    mejor_fit::Float64
    hist_mejor::Vector{Float64}   # mejor fitness por generación
    hist_prom::Vector{Float64}    # fitness promedio de la población por generación
    hist_evals::Vector{Int}       # evaluaciones acumuladas al final de cada generación
    evals::Int
    tiempo::Float64               # segundos
end

# =============================================================================
# 5. GA PRINCIPAL
# =============================================================================

function ejecutar_ga(fitness_fn::Function; semilla::Int,
                     params::ParamsGA = ParamsGA(),
                     p_max::Float64 = P_MAX, n_vars::Int = N_VARS)
    t0 = time()
    rng = MersenneTwister(semilla)
    p = params
    sigma_abs = p.sigma_rel * p_max
    n_hijos_gen = p.tam_pob - p.n_elite

    # Población inicial uniforme en [0, p_max]
    pob = [p_max .* rand(rng, n_vars) for _ in 1:p.tam_pob]
    fit = [Float64(fitness_fn(x)) for x in pob]
    evals = p.tam_pob

    hist_mejor = Float64[]; hist_prom = Float64[]; hist_evals = Int[]
    registrar!() = (push!(hist_mejor, maximum(fit));
                    push!(hist_prom, mean(fit));
                    push!(hist_evals, evals))
    registrar!()   # generación 0

    # Cada generación consume exactamente n_hijos_gen evaluaciones
    while evals + n_hijos_gen <= p.max_evals
        orden = sortperm(fit, rev = true)
        nueva = [copy(pob[orden[i]]) for i in 1:p.n_elite]   # elitismo
        fit_nueva = fit[orden[1:p.n_elite]]

        while length(nueva) < p.tam_pob
            i = torneo(rng, fit, p.k_torneo)
            j = torneo(rng, fit, p.k_torneo)
            if rand(rng) < p.prob_cruce
                h1, h2 = cruce_blx(rng, pob[i], pob[j], p.alpha, p_max)
            else
                h1, h2 = copy(pob[i]), copy(pob[j])
            end
            for h in (h1, h2)
                length(nueva) >= p.tam_pob && break
                mutar!(rng, h, p.prob_mut, sigma_abs, p_max)
                push!(nueva, h)
                push!(fit_nueva, Float64(fitness_fn(h)))
                evals += 1
            end
        end
        pob, fit = nueva, fit_nueva
        registrar!()
    end

    ib = argmax(fit)
    return ResultadoGA(semilla, copy(pob[ib]), fit[ib],
                       hist_mejor, hist_prom, hist_evals, evals, time() - t0)
end

ejecutar_ga (generic function with 1 method)

## 3. Pruebas y resultados

### 3.1. Prueba de la lógica del GA

La primera prueba utiliza `fitness_dummy`. Se comprueban:

- que todas las potencias permanezcan dentro de `[0, P_MAX]`;
- que existan 100 variables;
- que no se supere el presupuesto de evaluaciones;
- que el mejor fitness no disminuya gracias al elitismo;
- que una misma semilla produzca resultados reproducibles.


In [30]:
r_dummy = ejecutar_ga(fitness_dummy; semilla=42)

@assert all(0.0 .<= r_dummy.mejor_x .<= P_MAX)
@assert length(r_dummy.mejor_x) == N_VARS
@assert r_dummy.evals <= ParamsGA().max_evals
@assert all(diff(r_dummy.hist_mejor) .>= 0)
@assert r_dummy.hist_mejor[end] > r_dummy.hist_mejor[1]

r_dummy_2 = ejecutar_ga(fitness_dummy; semilla=42)
@assert r_dummy.mejor_fit == r_dummy_2.mejor_fit

println("[OK] Prueba del GA con función sustituta")
println("Mejor fitness = ", r_dummy.mejor_fit)
println("Óptimo teórico = 0.0")


[OK] Prueba del GA con función sustituta
Mejor fitness = -0.002977114020357918
Óptimo teórico = 0.0


### 3.2. Prueba con la función fitness real

Se comprueban dos casos de referencia:

- todas las antenas apagadas;
- todas las antenas funcionando a potencia máxima.

Después se ejecuta el GA con 50 000 evaluaciones para comprobar la integración con el modelo real.


In [31]:
f_real = x -> Fit.fitness(x; lambda=LAMBDA_DEFAULT)

println("Fitness con todas las antenas apagadas: ",
        f_real(zeros(N_VARS)))

println("Fitness con todas las antenas a potencia máxima: ",
        f_real(fill(P_MAX, N_VARS)))

r_real = ejecutar_ga(f_real; semilla=1, params=ParamsGA(max_evals=50_000))

@assert abs(r_real.mejor_fit - f_real(r_real.mejor_x)) < 1e-12

println()
println("[OK] Integración con fitness real")
println("Mejor fitness después de 50 000 evaluaciones = ",
        round(r_real.mejor_fit, digits=4))
println("Evaluaciones realizadas = ", r_real.evals)
println("Tiempo = ", round(r_real.tiempo, digits=2), " s")


Fitness con todas las antenas apagadas: 0.0
Fitness con todas las antenas a potencia máxima: 0.5

[OK] Integración con fitness real
Mejor fitness después de 50 000 evaluaciones = 0.9909
Evaluaciones realizadas = 49970
Tiempo = 5.99 s


### 3.3. Convergencia

In [32]:
println("Generaciones registradas: ", length(r_real.hist_mejor))
println("Fitness inicial: ", r_real.hist_mejor[1])
println("Fitness final: ", r_real.hist_mejor[end])


Generaciones registradas: 1041
Fitness inicial: 0.8669671022567359
Fitness final: 0.9908927245061642


### 3.4. Experimento con varias semillas

El siguiente bloque reproduce el experimento principal sin depender de la exportación CSV.


In [33]:
semillas = [42, 123, 256, 789, 1024]
resultados = ResultadoGA[]

for s in semillas
    r = ejecutar_ga(f_real; semilla=s)
    push!(resultados, r)
    @printf("Semilla %d | fitness = %.6f | evals = %d | tiempo = %.2f s\n",
            s, r.mejor_fit, r.evals, r.tiempo)
end

fits = [r.mejor_fit for r in resultados]

println()
println("Mejor fitness: ", maximum(fits))
println("Promedio: ", mean(fits))
println("Desviación estándar: ", length(fits) > 1 ? std(fits) : 0.0)


Semilla 42 | fitness = 0.989271 | evals = 49970 | tiempo = 5.90 s
Semilla 123 | fitness = 0.989959 | evals = 49970 | tiempo = 5.79 s
Semilla 256 | fitness = 0.986988 | evals = 49970 | tiempo = 5.65 s
Semilla 789 | fitness = 0.990412 | evals = 49970 | tiempo = 5.66 s
Semilla 1024 | fitness = 0.989890 | evals = 49970 | tiempo = 5.66 s

Mejor fitness: 0.9904122568081491
Promedio: 0.9893041715040237
Desviación estándar: 0.0013568540994260251


### 3.5. Exportación de resultados a CSV

In [34]:
RUTA_SALIDA = normpath(joinpath(@__DIR__, "..", "analytics", "output"))
mkpath(RUTA_SALIDA)

function _guardar_csv(ruta::String, cabecera::Vector{String}, filas::AbstractMatrix)
    writedlm(ruta, vcat(permutedims(cabecera), filas), ',')
end

tag = "lambda_$(LAMBDA_DEFAULT)"

# 1. Curva de convergencia individual de cada semilla
for r in resultados
    _guardar_csv(
        joinpath(RUTA_SALIDA, "ga_convergencia_semilla_$(r.semilla)_$(tag).csv"),
        ["Generacion", "Evaluaciones", "Mejor_Fitness", "Fitness_Promedio_Poblacion"],
        hcat(
            0:length(r.hist_mejor)-1,
            r.hist_evals,
            r.hist_mejor,
            r.hist_prom
        )
    )
end

# 2. Curva de convergencia promedio entre semillas
G = minimum(length(r.hist_mejor) for r in resultados)
M = reduce(hcat, [r.hist_mejor[1:G] for r in resultados])
prom = vec(mean(M, dims=2))
desv = size(M, 2) > 1 ? vec(std(M, dims=2)) : zeros(G)

_guardar_csv(
    joinpath(RUTA_SALIDA, "ga_convergencia_$(tag).csv"),
    ["Generacion", "Evaluaciones", "Fitness_Promedio", "Fitness_Std"],
    hcat(
        0:G-1,
        resultados[1].hist_evals[1:G],
        prom,
        desv
    )
)

# 3. Potencias finales de todas las semillas y antenas
n = length(resultados) * N_VARS
pot = Matrix{Any}(undef, n, 7)
fila = 0

for r in resultados, j in 1:N_VARS
    fila += 1
    pot[fila, :] = Any[
        "GA",
        LAMBDA_DEFAULT,
        r.semilla,
        j,
        ANTENAS[j][1],
        ANTENAS[j][2],
        r.mejor_x[j]
    ]
end

_guardar_csv(
    joinpath(RUTA_SALIDA, "ga_potencias_$(tag).csv"),
    ["Algoritmo", "Lambda", "Semilla", "Antena_ID", "Pos_X", "Pos_Y", "Potencia_W"],
    pot
)

# 4. Resumen de cada corrida
res = Matrix{Any}(undef, length(resultados), 9)

for (k, r) in enumerate(resultados)
    d = Fit.evaluar_solucion(r.mejor_x; lambda=LAMBDA_DEFAULT)

    @assert abs(d.fitness - r.mejor_fit) < 1e-9

    res[k, :] = Any[
        "GA",
        LAMBDA_DEFAULT,
        r.semilla,
        r.mejor_fit,
        d.cobertura_porcentaje,
        d.interferencia_norm,
        d.potencia_total_watts,
        r.evals,
        r.tiempo
    ]
end

_guardar_csv(
    joinpath(RUTA_SALIDA, "ga_resumen_$(tag).csv"),
    ["Algoritmo", "Lambda", "Semilla", "Mejor_Fitness",
     "Cobertura_pct", "Interferencia_norm", "Potencia_total_W",
     "Evaluaciones", "Tiempo_s"],
    res
)

println("CSV guardados en: ", RUTA_SALIDA)
println(" - ga_convergencia_$(tag).csv")
println(" - ga_convergencia_semilla_<semilla>_$(tag).csv")
println(" - ga_potencias_$(tag).csv")
println(" - ga_resumen_$(tag).csv")


CSV guardados en: c:\Users\USER\Desktop\Trabajos\5g-rural-coverage-heuristics\analytics\output
 - ga_convergencia_lambda_0.5.csv
 - ga_convergencia_semilla_<semilla>_lambda_0.5.csv
 - ga_potencias_lambda_0.5.csv
 - ga_resumen_lambda_0.5.csv


### 3.6. Mejor solución encontrada

Finalmente se obtiene la configuración de potencias correspondiente a la mejor corrida.


In [22]:
ib = argmax(fits)
mejor = resultados[ib]

println("Semilla ganadora: ", mejor.semilla)
println("Mejor fitness: ", mejor.mejor_fit)
println("Potencia total: ", sum(mejor.mejor_x), " W")
println("Potencia mínima: ", minimum(mejor.mejor_x), " W")
println("Potencia máxima: ", maximum(mejor.mejor_x), " W")
println("Primeras 10 potencias:")
println(mejor.mejor_x[1:10])


Semilla ganadora: 789
Mejor fitness: 0.9891652542946118
Potencia total: 10.765462988310452 W
Potencia mínima: 0.0 W
Potencia máxima: 0.6546916903888564 W
Primeras 10 potencias:
[0.07932452391837563, 0.19912059991846154, 0.0048025290005244544, 0.39427782177811366, 0.0, 0.1261321490664185, 0.2796294683705099, 0.0035025542960301173, 0.20557661815555373, 0.22327161698649525]


## 4. Conclusiones

El Algoritmo Genético implementado permite representar la configuración de potencia de las 100 antenas como un cromosoma de variables reales y buscar configuraciones que maximicen la función objetivo definida para la red 5G rural.

La implementación incorpora selección por torneo, cruce BLX-α, mutación gaussiana, reparación de límites y elitismo. Además, el uso de semillas y un presupuesto explícito de evaluaciones permite realizar experimentos reproducibles y comparar posteriormente el GA con otras heurísticas.
